# Extensões: placebo, few-shot por analogia e reescrita estruturada

Este notebook **não altera nada** do `analise_jev.ipynb` nem refaz o que já rodou: só lê os resultados e caches
que já existem e grava os seus próprios arquivos (nomes novos).

**Parte 1: o cartão ajuda por informação ou só por "ter contexto"? E estimar por analogia ajuda?**
Lê as predições do `30_run_llm.py` (nenhuma chamada de LLM aqui) e compara, nas mesmas issues:
- `ZS1` × `ZS1 + cartão` × `ZS1 + placebo` (cartão de **outro** projeto do mesmo dataset). Se o cartão real não
  ganhar do placebo, o ganho do cartão é calibração genérica, não informação do projeto.
- few-shot com exemplos do treino do **mesmo projeto**: sorteados × **os mais parecidos** (TF-IDF), com e sem
  cartão. É o que um time faz ao estimar por analogia; a mediana do projeto é a referência a vencer.

**Parte 2: reescrita estruturada pelos fatores de esforço** (motivação: Rasheed et al., 2026, *Automated Software
Engineering* 33:60, mostram que tamanho, complexidade, tecnologias, linguagem e plataforma são os fatores que mais
explicam o esforço, e deixam como trabalho futuro extraí-los do texto com LLM).
- Nova versão do texto, **estruturar_contexto**: o LLM reescreve a issue num modelo fixo (tipo, objetivo, escopo,
  componentes afetados, tecnologias, dependências externas, critérios de aceite, dúvidas em aberto), usando só a
  issue e o cartão do projeto, e escreve **"Not stated"** quando a informação não existe. Tamanho e complexidade
  ficam de fora de propósito: seria o LLM estimando disfarçado.
- Entra no fatorial da seção 12 como 5ª versão (× cartão sim/não no estimador), nas mesmas issues.
- **Lacunas:** quantos campos ficaram "Not stated" mede o que falta nas issues reais, e se isso se relaciona com o
  erro.

**Ordem para rodar** (os comandos do `30_run_llm.py` podem rodar enquanto você faz outra coisa; não rode dois
processos no mesmo Ollama ao mesmo tempo, fica lento para os dois):
1. `analise_jev.ipynb` até a seção 12 (a parte 2 usa os caches dela).
2. Os comandos da parte 1 (célula de configuração abaixo).
3. Este notebook, de cima a baixo. Cada parte pula sozinha o que ainda não tiver os arquivos.

In [ ]:
# ============================== CONFIGURAÇÃO ==============================
DATASETS = ["agile_limpo", "neo_limpo"]
LLM = "llama3"                     # nome do modelo usado no 30_run_llm.py (prefixo dos arquivos de predição)
ENGINE = "jev"; SUBSET = "test"     # os mesmos do analise_jev.ipynb

# Parte 2 (os mesmos do analise_jev.ipynb)
REWRITER = {"provider": "ollama", "model": "llama3"}
REWRITER_RPM = 0
ESTIMADOR_LLM = {"provider": "ollama", "model": "llama3"}
ESTIMADORES_ML = ["KNN", "XGBoost"]
RODAR_PARTE_2 = True               # False = só a parte 1 (não chama LLM nem Jev)

# Chaves de API: se faltarem, o notebook pergunta na hora (campo de senha, não fica salvo no arquivo).
# NÃO escreva a chave do Jev aqui.

# ---- Comandos da parte 1 (rode no terminal, da pasta do repositório) ----
# 1) placebo (≈ 787 chamadas):
#    python repro/30_run_llm.py --provider ollama --model llama3 --dataset agile_limpo neo_limpo --cartao-placebo
# 2) few-shot com os exemplos mais parecidos do mesmo projeto (≈ 787):
#    python repro/30_run_llm.py --provider ollama --model llama3 --dataset agile_limpo neo_limpo --protocol within --template FS1 --shots-from similar_same_project --so-projetos-com-cartao
# 3) o mesmo + cartão (≈ 787):
#    python repro/30_run_llm.py --provider ollama --model llama3 --dataset agile_limpo neo_limpo --protocol within --template FS1 --shots-from similar_same_project --cartao
# 4) controle: exemplos do mesmo projeto SORTEADOS (≈ 787):
#    python repro/30_run_llm.py --provider ollama --model llama3 --dataset agile_limpo neo_limpo --protocol within --template FS1 --shots-from same_project --so-projetos-com-cartao

In [ ]:
import sys, os, re, json, time, hashlib, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display as _ipy_display

ROOT = Path.cwd()
while not (ROOT / "see" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise SystemExit("Abra o notebook de dentro da pasta repro/.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from see import config
from see.io import load_merged
from see.quality import add_quality_scores
from see.text import clean_light
from see.prompts import truncate, zero_shot, parse_estimate

EXP = config.RESULTS / "improvement"            # caches da seção 12 (lidos) e da parte 2 (nomes novos)
OUT = config.RESULTS / "analise_extensoes"       # tabelas e figuras deste notebook
FIGDIR = OUT / "figuras"
for d in (EXP, OUT, FIGDIR):
    d.mkdir(parents=True, exist_ok=True)

TABELAS, _nfig = {}, [0]
def display(obj, nome=None):
    if nome is not None and isinstance(obj, (pd.DataFrame, pd.Series)):
        TABELAS[nome] = obj.to_frame() if isinstance(obj, pd.Series) else obj.copy()
    _ipy_display(obj)

def salva_fig(nome):
    _nfig[0] += 1
    plt.gcf().savefig(FIGDIR / f"{_nfig[0]:02d}_{nome}.png", dpi=150, bbox_inches="tight")

def load_script(name):
    spec = importlib.util.spec_from_file_location(name.replace(".py", "").lstrip("0123456789_"), ROOT / name)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); return mod

def pedir_chave(var, para):
    if not os.environ.get(var):
        import getpass
        os.environ[var] = getpass.getpass(f"{var} ({para}): ").strip()
    return bool(os.environ.get(var))

def wilcoxon_pareado(a, b):
    d = (b - a).dropna()
    if len(d) < 5 or (d == 0).all():
        return {"n": len(d), "Δ média": d.mean(), "p (Wilcoxon)": np.nan, "r (rank-biserial)": np.nan}
    w = stats.wilcoxon(d)
    ranks = d.abs().rank()
    return {"n": len(d), "Δ média": d.mean(), "p (Wilcoxon)": w.pvalue,
            "r (rank-biserial)": (ranks[d > 0].sum() - ranks[d < 0].sum()) / ranks.sum()}

def bootstrap_ci(d, n=5000, seed=0):
    rng = np.random.default_rng(seed); d = np.asarray(pd.Series(d).dropna())
    if len(d) < 5:
        return (np.nan, np.nan)
    m = rng.choice(d, (n, len(d)), replace=True).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))

def holm(p):
    # correção de Holm dentro de uma família de contrastes (NaN fica NaN)
    p = pd.Series(p, dtype=float); ok = p.dropna().sort_values()
    m = len(ok); adj = pd.Series(np.nan, index=p.index)
    run = 0.0
    for k, (i, v) in enumerate(ok.items()):
        run = max(run, min(1.0, (m - k) * v)); adj[i] = run
    return adj

TRAIN_MED = {ds: float(load_merged(ds).query("within == 'train'").sp.median()) for ds in DATASETS}

COR = {"s1": "#2a78d6", "s2": "#eb6834", "s3": "#1baf7a", "s4": "#8a5cd6", "ink": "#0b0b0b", "ink2": "#52514e",
       "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7", "surface": "#fcfcfb"}
plt.rcParams.update({
    "figure.facecolor": COR["surface"], "axes.facecolor": COR["surface"], "savefig.facecolor": COR["surface"],
    "axes.edgecolor": COR["axis"], "axes.labelcolor": COR["ink2"], "xtick.color": COR["muted"], "ytick.color": COR["muted"],
    "axes.grid": True, "grid.color": COR["grid"], "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.titlecolor": COR["ink"], "font.size": 9.5, "legend.frameon": False, "figure.dpi": 110,
})
pd.set_option("display.float_format", "{:.3f}".format); pd.set_option("display.width", 160)

---
# Parte 1 — Cartão × placebo e few-shot por analogia

Só lê `results/predictions/{dataset}/...` (gerados pelo `30_run_llm.py`). Todas as comparações usam **as mesmas
issues**: as do test set (`within_eval`) de projetos com cartão, presentes em todas as configurações disponíveis.
Estimativa sem número → mediana do treino (como no `90_evaluate.py`). Os p-valores têm correção de Holm dentro de
cada dataset.

In [ ]:
CONFIGS = {   # nome legível -> (protocolo, arquivo)
    "ZS1":                         ("within", f"{LLM}__ZS1.csv"),
    "ZS1 + cartão":                ("within", f"{LLM}__ZS1__cartao.csv"),
    "ZS1 + placebo":               ("within", f"{LLM}__ZS1__cartaoplacebo.csv"),
    "FS mesmo projeto, sorteados": ("within_fewshot_sameproj", f"{LLM}__FS1__same_project.csv"),
    "FS mesmo projeto, parecidos": ("within_fewshot_simproj", f"{LLM}__FS1__similar_same_project.csv"),
    "FS parecidos + cartão":       ("within_fewshot_simproj", f"{LLM}__FS1__similar_same_project__cartao.csv"),
}
REFERENCIAS = {"ProjectMedian": ("within", "ProjectMedian.csv"), "SVR": ("within", "SVR.csv"),
               "XGBoost": ("within", "XGBoost.csv")}

P1, disp = {}, []
for ds in DATASETS:
    for nome, (proto, arq) in {**CONFIGS, **REFERENCIAS}.items():
        f = config.PREDICTIONS / ds / proto / arq
        ok = f.exists()
        disp.append({"dataset": ds, "configuração": nome, "arquivo": f"{proto}/{arq}", "existe": ok,
                     "issues": len(pd.read_csv(f)) if ok else 0})
        if ok:
            p = pd.read_csv(f, dtype={"issue_id": str, "project": str})
            n_test = int((load_merged(ds).within == "test").sum())
            if nome in REFERENCIAS and len(p) != n_test:      # ex.: SVR treinado no dataset errado
                print(f"⚠ {ds}/{proto}/{arq} tem {len(p)} linhas, mas o test set tem {n_test}: fica de fora "
                      f"(rode de novo: python 10_run_ml.py --dataset {ds} ...)")
                disp[-1]["existe"] = "tamanho errado"; continue
            P1[(ds, nome)] = p.set_index("issue_id")
display(pd.DataFrame(disp).set_index(["dataset", "configuração"]), nome="P1 arquivos")
print("Configurações que faltam ficam de fora; rode os comandos da célula de configuração e volte aqui.")

In [ ]:
SECAO1 = []
rows_mae, rows_c, por_proj = [], [], []
CONTRASTES_1 = [
    ("ZS1", "ZS1 + cartão", "efeito do cartão"),
    ("ZS1", "ZS1 + placebo", "efeito do placebo"),
    ("ZS1 + placebo", "ZS1 + cartão", "informação do cartão (cartão × placebo)"),
    ("ZS1", "FS mesmo projeto, sorteados", "exemplos do mesmo projeto"),
    ("FS mesmo projeto, sorteados", "FS mesmo projeto, parecidos", "escolher os parecidos"),
    ("FS mesmo projeto, parecidos", "FS parecidos + cartão", "cartão junto com os exemplos"),
    ("ZS1 + cartão", "FS parecidos + cartão", "exemplos parecidos, com cartão nos dois"),
    ("ProjectMedian", "ZS1 + cartão", "LLM + cartão × mediana do projeto"),
    ("ProjectMedian", "FS mesmo projeto, parecidos", "analogia × mediana do projeto"),
    ("ProjectMedian", "FS parecidos + cartão", "analogia + cartão × mediana do projeto"),
]
E1 = {}
for ds in DATASETS:
    llm_ok = [n for n in CONFIGS if (ds, n) in P1]
    if not llm_ok:
        print(f"[{ds}] nenhuma predição de LLM encontrada"); continue
    ids = set.intersection(*[set(P1[(ds, n)].index) for n in llm_ok])
    ids &= set(P1[(ds, "ZS1 + cartão")].index) if (ds, "ZS1 + cartão") in P1 else ids
    ids = sorted(ids)
    y = P1[(ds, llm_ok[0])].loc[ids, "y_true"].astype(float)
    proj = load_merged(ds).set_index("issue_id").loc[ids, "project"].astype(str)
    E = pd.DataFrame(index=ids)
    for n in [*CONFIGS, *REFERENCIAS]:
        if (ds, n) in P1 and set(ids) <= set(P1[(ds, n)].index):
            pred = P1[(ds, n)].loc[ids, "y_pred"].astype(float)
            E[n] = (y - pred.fillna(TRAIN_MED[ds])).abs()
            rows_mae.append({"dataset": ds, "configuração": n, "MAE": E[n].mean(), "MdAE": E[n].median(),
                             "sem número": int(pred.isna().sum()), "n": len(ids)})
    E1[ds] = E
    por_proj.append(E.groupby(proj.values).mean().assign(dataset=ds, n=E.groupby(proj.values).size()))
    fam = []
    for a, b, desc in CONTRASTES_1:
        if a in E and b in E:
            r = wilcoxon_pareado(E[a], E[b]); lo, hi = bootstrap_ci(E[b] - E[a])
            fam.append({"dataset": ds, "contraste": desc, "A": a, "B": b, "MAE A": E[a].mean(), "MAE B": E[b].mean(),
                        "Δ MAE (B − A)": r["Δ média"], "IC95% inf": lo, "IC95% sup": hi,
                        "p (Wilcoxon)": r["p (Wilcoxon)"], "r (rank-biserial)": r["r (rank-biserial)"], "n": r["n"]})
    fam = pd.DataFrame(fam)
    if len(fam):
        fam["p (Holm)"] = holm(fam["p (Wilcoxon)"]).values
        rows_c.append(fam)

if rows_mae:
    MAE1 = pd.DataFrame(rows_mae).set_index(["dataset", "configuração"])
    display(MAE1, nome="P1 MAE nas mesmas issues")
if rows_c:
    C1 = pd.concat(rows_c).set_index(["dataset", "contraste"])
    C1.to_csv(OUT / "parte1_contrastes.csv")
    display(C1, nome="P1 contrastes")
    print("Δ negativo = B errou menos que A. A linha-chave do placebo é 'informação do cartão (cartão × placebo)':")
    print("se o IC95% cruza o zero, não dá para dizer que o cartão traz mais que um texto de contexto qualquer.")
if por_proj:
    PP1 = pd.concat(por_proj).rename_axis("projeto").reset_index().set_index(["dataset", "projeto"])
    display(PP1.round(2), nome="P1 MAE por projeto")

In [ ]:
if rows_mae:
    ordem = [n for n in [*CONFIGS, "SVR", "XGBoost"] if n in MAE1.index.get_level_values(1)]
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.6 * len(DATASETS), 0.42 * len(ordem) + 1.4))
    for ax, ds in zip(np.atleast_1d(axes), DATASETS):
        if ds not in MAE1.index.get_level_values(0):
            ax.set_visible(False); continue
        sub = MAE1.loc[ds].reindex(ordem).dropna(subset=["MAE"])
        cores = [COR["s2"] if "cartão" in n else COR["muted"] if "placebo" in n else
                 COR["axis"] if n in REFERENCIAS else COR["s1"] for n in sub.index]
        ax.barh(range(len(sub)), sub.MAE, color=cores, height=0.65)
        for i, v in enumerate(sub.MAE):
            ax.text(v, i, f" {v:.2f}", va="center", fontsize=8, color=COR["ink2"])
        if "ProjectMedian" in MAE1.loc[ds].index:
            pm = MAE1.loc[(ds, "ProjectMedian"), "MAE"]
            ax.axvline(pm, color=COR["ink2"], linestyle="--", linewidth=1)
            ax.text(pm, len(sub) - 0.4, " mediana do projeto", fontsize=8, color=COR["ink2"])
        ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.index); ax.invert_yaxis()
        ax.set_xlabel("MAE (story points)"); ax.set_title(f"{ds} (n = {int(sub.n.iloc[0])})"); ax.grid(axis="y", visible=False)
    fig.suptitle("Mesmas issues: zero-shot, cartão, placebo e few-shot do mesmo projeto", fontweight="bold", color=COR["ink"])
    fig.tight_layout(); salva_fig("parte1_mae"); plt.show()

---
# Parte 2 — Reescrita estruturada pelos fatores de esforço

Usa as issues da seção 12 do `analise_jev.ipynb` (`results/improvement/selecionadas_contexto.csv`) que têm as
8 estimativas do fatorial (4 versões × cartão sim/não, caches `llm_{modelo}_{versão}[__cartao].jsonl`). Nada disso
é refeito: só a versão nova é gerada, julgada pelo Jev e estimada. Caches novos:
`rewrites_estruturar_contexto_*.jsonl`, `jev_estruturar_contexto.jsonl`, `llm_{modelo}_estruturar_contexto[__cartao].jsonl`
e `pred_{dataset}_{modelo}_estruturar.csv`.

In [ ]:
VERSOES_12 = ["original", "parafrasear", "melhorar", "melhorar_contexto"]
NOVA = "estruturar_contexto"
SLUG_EST = re.sub(r'[^A-Za-z0-9._-]', '-', ESTIMADOR_LLM["model"]) if ESTIMADOR_LLM else None

def le_cache_llm(versao, com_cartao):
    path = EXP / f"llm_{SLUG_EST}_{versao}{'__cartao' if com_cartao else ''}.jsonl"
    done = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line); done[r["issue_id"]] = r
    return path, done

PARTE2 = RODAR_PARTE_2 and ESTIMADOR_LLM is not None
sel_path = EXP / "selecionadas_contexto.csv"
if PARTE2 and not sel_path.exists():
    print("Falta results/improvement/selecionadas_contexto.csv: rode a seção 12 do analise_jev.ipynb antes."); PARTE2 = False
if PARTE2:
    SELC = pd.read_csv(sel_path, dtype={"issue_id": str, "project": str})
    SELC = SELC[SELC.dataset.isin(DATASETS)]
    ids2 = set(SELC.issue_id)
    for v in VERSOES_12:
        for c in (False, True):
            ids2 &= set(le_cache_llm(v, c)[1])
    print(f"{len(ids2)} das {len(SELC)} issues da seção 12 têm as 8 estimativas do fatorial "
          f"({SELC[SELC.issue_id.isin(ids2)].groupby('dataset').size().to_dict()})")
    if not ids2:
        print("Nenhuma: rode a seção 12 (12d) do analise_jev.ipynb até o fim antes."); PARTE2 = False

if PARTE2:
    # texto original e notas do Jev das issues (os mesmos arquivos que o analise_jev.ipynb lê)
    QUESTIONS = json.loads((ROOT / "quality_criteria.json").read_text(encoding="utf-8"))
    QUESTIONS = {k: v for k, v in QUESTIONS.items() if not k.startswith("_")}
    frames = []
    for ds in DATASETS:
        j = add_quality_scores(pd.read_csv(config.RESULTS / "quality" / f"{ds}{'' if SUBSET == 'eval' else '_' + SUBSET}__{ENGINE}.csv",
                                           dtype={"issue_id": str, "project": str}))
        data = load_merged(ds)
        j = j.drop(columns=["project", "sp"], errors="ignore").merge(
            data[["issue_id", "project", "sp", "title", "description", "within"]], on="issue_id", how="left")
        j["dataset"] = ds
        frames.append(j)
    J = pd.concat(frames, ignore_index=True)
    J = J[J.issue_id.isin(ids2)]
    CARDS = pd.read_csv(config.DATA_PROCESSED / "project_cards.csv", dtype={"project": str}).fillna("")
    CARD = {(r.dataset, r.project): (r.cartao or r.cartao_bruto) for r in CARDS.itertuples()}
    cartao = lambda ds, p: CARD.get((config.base_dataset(ds), str(p)))
    ORIG = J[["issue_id", "dataset", "project", "sp", "title", "description"]].copy()
    ORIG["title"] = ORIG.title.map(clean_light); ORIG["description"] = ORIG.description.fillna("").map(clean_light)
    ORIG = ORIG.sort_values(["dataset", "issue_id"]).reset_index(drop=True)

In [ ]:
# ---- 2a. reescrita estruturada (com o cartão do projeto) ----
from see.llm_clients import LLMClient, call_with_retry

CAMPOS = ["Type", "Goal", "Scope", "Affected components", "Technologies", "External dependencies",
          "Acceptance criteria", "Open questions"]
CAMPOS_LACUNA = ["Goal", "Scope", "Affected components", "Technologies", "Acceptance criteria"]   # informação da issue

PROMPT_ESTRUTURAR = '''You are a Quality Analyst on an agile team. Rewrite the work item below into the structured template that follows. The template lists the information an agile team needs to estimate a work item (effort drivers). You also receive background about the project, as a member of the team would know it.

Rules:
- Use ONLY information stated in the original work item or directly supported by the project context. Do NOT invent features, requirements, numbers, components or acceptance criteria.
- When the information for a field is not available, write exactly "Not stated". Do not guess.
- Preserve every technical detail of the original (error messages, file names, versions, steps to reproduce).
- Never mention effort, size, complexity, time or story points.

Template for the description (keep the field names, one field per line, in this order):
Type: <bug | feature | improvement | technical task | documentation | Not stated>
Goal: <what must be achieved and why, in one or two sentences>
Scope: <what has to be changed or built; list the subtasks if the original implies them>
Affected components: <modules, screens, services, APIs or files involved>
Technologies: <languages, frameworks, libraries, databases or platforms involved>
External dependencies: <other teams, third-party services, hardware or upstream issues>
Acceptance criteria: <verifiable conditions, only if they can be inferred>
Open questions: <what is unclear or missing in the original>

Project context:
{context}

Original work item:
Title: {title}
Description: {description}

Respond ONLY with valid JSON: {{"title": "...", "description": "<the filled template>", "justification": "..."}}'''

PLACEHOLDER = re.compile(r"^\s*(\.{2,}|…|<[^>]*>|n/?a|none|null|tbd)?\s*$", re.I)
FORBIDDEN = re.compile(r"story ?points?|\bpoints?\b|\bhours?\b|\bdays?\b|effort|estimat", re.I)
_ROTULO = re.compile(r"(?im)^[\s*#>\-]*(" + "|".join(re.escape(c) for c in CAMPOS) + r")\s*\**\s*:")

def campos(desc):
    # separa a descrição estruturada nos campos do modelo: {campo: texto}
    marcas = [(m.start(), m.end(), m.group(1).lower()) for m in _ROTULO.finditer(desc or "")]
    out = {}
    for k, (ini, fim, nome) in enumerate(marcas):
        prox = marcas[k + 1][0] if k + 1 < len(marcas) else len(desc)
        nome = next(c for c in CAMPOS if c.lower() == nome)
        out.setdefault(nome, desc[fim:prox].strip(" \n*-"))
    return out

def nao_informado(txt):
    return txt is None or not re.sub(r"[\s.*_\-]", "", txt) or bool(re.fullmatch(r"\W*not stated\W*", txt.strip(), re.I))

def parse_json(text):
    text = re.sub(r"<think>.*?</think>", " ", text or "", flags=re.S)
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0), strict=False)       # aceita quebra de linha dentro das strings
    except json.JSONDecodeError:
        return None
    if isinstance(d.get("description"), dict):          # o modelo às vezes devolve os campos como objeto
        d["description"] = "\n".join(f"{k}: {v}" for k, v in d["description"].items())
    if isinstance(d.get("description"), list):
        d["description"] = "\n".join(map(str, d["description"]))
    return d if isinstance(d.get("title"), str) and isinstance(d.get("description"), str) else None

def valida(d, original_desc):
    if not d or PLACEHOLDER.match(d["title"]):
        return False
    if original_desc.strip() and PLACEHOLDER.match(d["description"]):
        return False
    return len(campos(d["description"])) >= 5            # seguiu o modelo (pelo menos 5 dos 8 campos)

def done_ids(path):
    out = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                if r.get("ok"):
                    out[r["issue_id"]] = r
    return out

_KEYS = {"gemini": "GEMINI_API_KEY", "groq": "GROQ_API_KEY", "openrouter": "OPENROUTER_API_KEY"}
if PARTE2:
    if REWRITER["provider"] in _KEYS:
        pedir_chave(_KEYS[REWRITER["provider"]], "quem reescreve")
    rewriter = LLMClient(REWRITER["provider"], REWRITER["model"], temperature=0.0, max_tokens=1500, seed=config.SEED)
    versao = hashlib.sha1((PROMPT_ESTRUTURAR + rewriter.model).encode()).hexdigest()[:8]
    REW_PATH = EXP / f"rewrites_{NOVA}_{re.sub(r'[^A-Za-z0-9._-]', '-', rewriter.model)}_{versao}.jsonl"
    done = done_ids(REW_PATH)
    todo = [r for r in ORIG.itertuples() if r.issue_id not in done]
    tentadas = set()
    if REW_PATH.exists():
        tentadas = {json.loads(l)["issue_id"] for l in REW_PATH.read_text(encoding="utf-8").splitlines() if l.strip()}
    print(f"{NOVA}: {len(done)} prontas, {len(todo)} a fazer ({len(set(i.issue_id for i in todo) & tentadas)} já falharam antes)")
    last = 0.0
    with open(REW_PATH, "a", encoding="utf-8") as f:
        for n, r in enumerate(todo, 1):
            if REWRITER_RPM:
                time.sleep(max(0, 60 / REWRITER_RPM - (time.time() - last)))
            last = time.time()
            orig_desc = truncate(r.description, 3000)
            prompt = PROMPT_ESTRUTURAR.format(context=cartao(r.dataset, r.project), title=r.title,
                                              description=orig_desc or "(no description)")
            rec = {"issue_id": r.issue_id, "condicao": NOVA, "model": rewriter.model}
            try:
                text, lat, _ = call_with_retry(rewriter, prompt)
                d = parse_json(text)
                rec.update(ok=valida(d, orig_desc), raw=text[:5000], latency_s=round(lat, 2), **(d or {}))
            except Exception as e:
                rec.update(ok=False, error=str(e)[:300])
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            if n % 10 == 0 or n == len(todo):
                print(f"   {n}/{len(todo)}")
    REW_E = done_ids(REW_PATH)
    print(f"{len(REW_E)}/{len(ORIG)} reescritas estruturadas válidas")

In [ ]:
# ---- 2b. checagens e lacunas ----
if PARTE2:
    ids_e = sorted(set(ORIG.issue_id) & set(REW_E))
    O = ORIG[ORIG.issue_id.isin(ids_e)].reset_index(drop=True)
    S = O.copy()
    S["title"] = S.issue_id.map(lambda i: REW_E[i]["title"])
    S["description"] = S.issue_id.map(lambda i: REW_E[i]["description"])
    VERS = {"original": O, NOVA: S}

    STOP = set(("a an the and or of to in on for with by from at as is are was were be been it this that these those "
                "not no but if then than so such can could should would will may might must do does did done has have "
                "had i we you they he she its our your their my me us them which who what when where how all any each "
                "some more most other into also only just there here up out over about after before while because "
                "type goal scope affected components technologies external dependencies acceptance criteria open "
                "questions stated").split())       # os rótulos do modelo não contam como palavra nova
    palavras = lambda t: {w for w in re.findall(r"[a-z][a-z0-9_]{2,}", (t or "").lower()) if w not in STOP}
    orig_w = {r.issue_id: palavras(r.title + " " + r.description) for r in O.itertuples()}
    card_w = {r.issue_id: palavras(cartao(r.dataset, r.project)) for r in O.itertuples()}
    chk = []
    for name, v in VERS.items():
        txt = v.title + " " + v.description
        novas = [len(palavras(t) - orig_w[i]) / max(1, len(palavras(t))) for i, t in zip(v.issue_id, txt)]
        usa = [bool((palavras(t) & card_w[i]) - orig_w[i]) for i, t in zip(v.issue_id, txt)]
        chk.append({"versão": name, "palavras (mediana)": txt.str.split().str.len().median(),
                    "menciona esforço/tempo/pontos_%": 100 * txt.str.contains(FORBIDDEN).mean(),
                    "palavras novas (mediana %)": 100 * np.median(novas),
                    "traz termo do cartão ausente do original_%": 100 * np.mean(usa)})
    display(pd.DataFrame(chk).set_index("versão").round(1), nome="P2 checagens")
    print("Compare com a tabela 'checagens com contexto' da seção 12b (melhorar_contexto).")

    # lacunas: campos que o LLM marcou como "Not stated"
    L = pd.DataFrame([{"issue_id": r.issue_id, "dataset": r.dataset, "project": r.project, "sp": r.sp,
                       **{c: nao_informado(campos(r.description).get(c)) for c in CAMPOS}} for r in S.itertuples()])
    L["n_lacunas"] = L[CAMPOS_LACUNA].sum(axis=1)
    LAC = (L.groupby("dataset")[CAMPOS].mean().mul(100).T.rename_axis("campo (% 'Not stated')"))
    display(LAC.round(1), nome="P2 lacunas por campo")
    display(L.groupby("dataset").n_lacunas.describe().round(2), nome="P2 lacunas por issue")
    print(f"n_lacunas conta {CAMPOS_LACUNA} (0 a {len(CAMPOS_LACUNA)}). 'External dependencies' e 'Open questions'")
    print("ficam de fora da contagem: 'Not stated' ali pode só significar que não há dependência / dúvida.")
    L.to_csv(OUT / "parte2_lacunas.csv", index=False)

    i0 = ids_e[0]
    r0 = O[O.issue_id == i0].iloc[0]
    print(f"\n===== ORIGINAL ({i0}, {r0.dataset}/{r0.project}, SP {r0.sp:g}) =====\nTITLE: {r0.title}\n{r0.description[:1000]}")
    r1 = S[S.issue_id == i0].iloc[0]
    print(f"\n===== {NOVA.upper()} =====\nTITLE: {r1.title}\n{r1.description[:1500]}")

In [ ]:
# ---- 2c. Jev: estruturar_contexto × original, melhorar e melhorar_contexto ----
if PARTE2:
    judge_mod = load_script("50_quality_judge.py")
    _judge = []
    def judge():
        if not _judge:
            if ENGINE == "jev":
                pedir_chave("TYPESAFE_API_KEY", "Jev")
            _judge.append(judge_mod.Judge(ENGINE, None, None))
        return _judge[0]

    def judge_version(name, v=None):
        """Lê o cache jev_{name}.jsonl (o mesmo da seção 10/12c); com v, julga as issues que faltam."""
        path = EXP / f"jev_{name}.jsonl"
        done = {}
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    r = json.loads(line)
                    if not r.get("error"):
                        done[r["issue_id"]] = r
        if v is not None:
            todo = v[~v.issue_id.isin(done)]
            print(f"Jev / {name}: {len(done)} prontas, {len(todo)} a fazer")
            with open(path, "a", encoding="utf-8") as f:
                for r in todo.itertuples():
                    state = f"Title: {r.title}\nDescription: {truncate(r.description, 2000) or '(no description)'}"
                    rec = {"issue_id": r.issue_id}
                    try:
                        resp, _ = judge().ask(state, QUESTIONS)
                        rec.update(answers=resp.get("answers", {}), error=None)
                    except Exception as e:
                        rec.update(answers={}, error=str(e)[:300])
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
                    if not rec["error"]:
                        done[r.issue_id] = rec
        rows = [{"issue_id": i, **judge_mod.flatten(rec["answers"], QUESTIONS)} for i, rec in done.items()]
        return add_quality_scores(pd.DataFrame(rows)) if rows else pd.DataFrame(columns=["issue_id"])

    JE = {"original": add_quality_scores(J[J.issue_id.isin(ids_e)][["issue_id"] + [c for c in J.columns
            if c.split("_conf")[0] in QUESTIONS or c.startswith("issue_type")]].copy())}
    for c in ["melhorar", "melhorar_contexto"]:
        JE[c] = judge_version(c)                           # só lê (já julgadas na seção 10/12c)
    JE[NOVA] = judge_version(NOVA, S)
    dims = [c for c in ["indice_qualidade", "nota_geral_juiz", "goal_clarity", "estimability",
                        "has_acceptance_criteria", "is_ambiguous", "scope_size"] if c in JE["original"]]
    meta = O.set_index("issue_id").dataset
    rows = []
    for a in ["original", "melhorar", "melhorar_contexto"]:
        m = (JE[a].set_index("issue_id")[dims].join(JE[NOVA].set_index("issue_id")[dims], rsuffix="_b", how="inner")
               .loc[lambda x: x.index.isin(ids_e)])
        for ds in DATASETS + ["ambos"]:
            mm = m if ds == "ambos" else m[m.index.map(meta) == ds]
            for d in dims:
                rows.append({"comparação": f"{NOVA} × {a}", "dataset": ds, "dimensão": d, "outra versão": mm[d].mean(),
                             NOVA: mm[f"{d}_b"].mean(), **wilcoxon_pareado(mm[d], mm[f"{d}_b"])})
    JEVE = pd.DataFrame(rows).set_index(["comparação", "dataset", "dimensão"])
    JEVE.to_csv(OUT / "parte2_jev.csv")
    display(JEVE.round(3), nome="P2 Jev")
    print("Δ média positivo em indice_qualidade = a estruturada ficou melhor que a outra versão.")

In [ ]:
# ---- 2d. Reestimativa: ML, mediana do projeto e LLM (com e sem cartão) ----
if PARTE2:
    ml = load_script("10_run_ml.py")
    EST = []
    for ds in DATASETS:
        ids = O[O.dataset == ds].issue_id
        if not len(ids):
            continue
        df = load_merged(ds); tr = df[df.within == "train"]
        for mname in ESTIMADORES_ML:
            old = EXP / f"pred_{ds}_{mname}_contexto.csv"            # 4 versões, seção 12d (só leitura)
            if old.exists():
                EST.append(pd.read_csv(old, dtype={"issue_id": str}))
            cache = EXP / f"pred_{ds}_{mname}_estruturar.csv"
            if not cache.exists():
                print(f"treinando {mname} ({ds}) no texto original (o mesmo modelo da 12d) ...")
                txt = S.set_index("issue_id").loc[ids].pipe(lambda v: (v.title + ". " + v.description).str.lower())
                x_tr, x_pr = ml.featurize(tr.text_full, txt, 0)
                model = ml.make_model(mname); model.fit(x_tr, tr.sp.to_numpy())
                pd.DataFrame({"issue_id": txt.index, "versao": NOVA, "modelo": mname, "dataset": ds,
                              "y_pred": model.predict(x_pr)}).to_csv(cache, index=False)
            EST.append(pd.read_csv(cache, dtype={"issue_id": str}))
        med = tr.groupby("project").sp.median()
        base = O[O.dataset == ds]
        for n in VERSOES_12 + [NOVA]:
            EST.append(pd.DataFrame({"issue_id": base.issue_id, "versao": n, "modelo": "ProjectMedian", "dataset": ds,
                                     "y_pred": base.project.map(med).values}))

    if ESTIMADOR_LLM["provider"] in _KEYS:
        pedir_chave(_KEYS[ESTIMADOR_LLM["provider"]], "LLM estimador")
    est_client = LLMClient(ESTIMADOR_LLM["provider"], ESTIMADOR_LLM["model"], temperature=0.0, seed=config.SEED)
    CARD_PREFIX = ("Project context (background about the software project this work item belongs to):\n"
                   "{card}\n\n")                     # o mesmo texto do 30_run_llm.py --cartao e da seção 12d
    tag0 = f"LLM:{ESTIMADOR_LLM['model']}"
    meta_ds = O.set_index("issue_id").dataset
    for com_cartao in (False, True):
        tag = tag0 + (" + cartão" if com_cartao else "")
        for n in VERSOES_12 + [NOVA]:
            path, done = le_cache_llm(n, com_cartao)
            if n == NOVA:                                 # só a versão nova chama o LLM
                todo = S[~S.issue_id.isin(done)]
                print(f"{tag} / {n}: {len(done)} prontas, {len(todo)} a fazer")
                with open(path, "a", encoding="utf-8") as f:
                    for r in todo.itertuples():
                        prompt = zero_shot("ZS1", r.title, truncate(r.description, 2000))
                        if com_cartao:
                            prompt = CARD_PREFIX.format(card=cartao(r.dataset, r.project)) + prompt
                        try:
                            text, _, _ = call_with_retry(est_client, prompt)
                            rec = {"issue_id": r.issue_id, "y_pred": parse_estimate(text), "raw": text[:200]}
                        except Exception as e:
                            rec = {"issue_id": r.issue_id, "y_pred": None, "error": str(e)[:200]}
                        f.write(json.dumps(rec) + "\n"); f.flush(); done[r.issue_id] = rec
            EST.append(pd.DataFrame({"issue_id": ids_e, "versao": n, "modelo": tag, "dataset": meta_ds.loc[ids_e].values,
                                     "y_pred": [done[i].get("y_pred") if i in done else None for i in ids_e]}))

    EST = pd.concat(EST, ignore_index=True)
    EST = EST[EST.issue_id.isin(ids_e)]
    EST["y_pred"] = pd.to_numeric(EST.y_pred, errors="coerce")
    inval = EST.y_pred.isna()
    if inval.any():
        print(f"{int(inval.sum())} estimativas sem número (LLM) → mediana do treino, como no 90_evaluate")
        EST.loc[inval, "y_pred"] = EST.loc[inval, "dataset"].map(TRAIN_MED)
    EE = EST.merge(O[["issue_id", "sp"]], on="issue_id")
    EE["abs_err"] = (EE.sp - EE.y_pred).abs()
    MAE2 = (EE.pivot_table(index=["dataset", "modelo"], columns="versao", values="abs_err", aggfunc="mean")
              .reindex(columns=VERSOES_12 + [NOVA]))
    display(MAE2.round(3), nome="P2 MAE por versão e modelo")
    print("ML e mediana do projeto: só versão do texto (o cartão não entra neles).")

### 2e. Fatorial com a 5ª versão: versão do texto × cartão no estimador (LLM)

| texto / prompt | sem cartão | com cartão |
|---|---|---|
| original | linha de base | só o contexto no estimador |
| parafrasear (controle) | só mexer no texto | |
| melhorar | reescrita livre sem contexto | |
| melhorar_contexto | reescrita livre com contexto | tudo junto (livre) |
| **estruturar_contexto** | **reescrita estruturada com contexto** | **tudo junto (estruturada)** |

O contraste novo que responde à pergunta é **estruturar_contexto × melhorar_contexto**: a mesma informação
(issue + cartão), em formato livre ou guiado pelos fatores de esforço. A regressão é a mesma da 12e
(`|erro| ~ versão × cartão + issue`, erro-padrão agrupado por issue), agora com 5 versões.

In [ ]:
if PARTE2:
    import statsmodels.formula.api as smf
    VERSOES = VERSOES_12 + [NOVA]
    F = EE[EE.modelo.isin([tag0, tag0 + " + cartão"])].copy()
    F["cartao"] = (F.modelo != tag0).astype(int)
    cont = F.groupby("issue_id").size()
    F = F[F.issue_id.isin(cont[cont == 2 * len(VERSOES)].index)]
    print(f"{F.issue_id.nunique()} issues com as {2 * len(VERSOES)} estimativas")

    tab = (F.pivot_table(index=["dataset", "versao"], columns="cartao", values="abs_err", aggfunc="mean")
             .rename(columns={0: "sem cartão", 1: "com cartão"}))
    tab = tab.reindex([(ds, v) for ds in DATASETS for v in VERSOES if (ds, v) in tab.index])
    tab["Δ do cartão"] = tab["com cartão"] - tab["sem cartão"]
    pm = EE[(EE.modelo == "ProjectMedian") & EE.issue_id.isin(F.issue_id) & (EE.versao == "original")].groupby("dataset").abs_err.mean()
    display(tab.round(3), nome="P2 fatorial MAE")
    print("Mediana do projeto nas mesmas issues:", pm.round(3).to_dict())

    PARES = [("original", 0, "original", 1, "só o contexto no estimador"),
             ("original", 0, "parafrasear", 0, "só mexer no texto (controle)"),
             ("original", 0, "melhorar", 0, "reescrita livre, sem contexto"),
             ("original", 0, "melhorar_contexto", 0, "reescrita livre com contexto"),
             ("original", 0, NOVA, 0, "reescrita estruturada com contexto"),
             ("melhorar_contexto", 0, NOVA, 0, "estruturada × livre (mesma informação)"),
             ("melhorar_contexto", 1, NOVA, 1, "estruturada × livre, com cartão no estimador"),
             ("original", 1, NOVA, 1, "estruturada, com cartão no estimador"),
             ("original", 0, "melhorar_contexto", 1, "tudo junto (livre)"),
             ("original", 0, NOVA, 1, "tudo junto (estruturada)")]
    W2 = F.pivot_table(index=["dataset", "issue_id"], columns=["versao", "cartao"], values="abs_err")
    nome_v = lambda v, c: v + (" + cartão" if c else "")
    rows = []
    for ds in DATASETS + ["ambos"]:
        g = W2 if ds == "ambos" else W2.xs(ds, level="dataset")
        fam = []
        for va, ca, vb, cb, desc in PARES:
            a, b = g[(va, ca)], g[(vb, cb)]
            r = wilcoxon_pareado(a, b); lo, hi = bootstrap_ci(b - a)
            fam.append({"dataset": ds, "contraste": desc, "A": nome_v(va, ca), "B": nome_v(vb, cb),
                        "MAE A": a.mean(), "MAE B": b.mean(), "Δ MAE (B − A)": r["Δ média"],
                        "IC95% inf": lo, "IC95% sup": hi, "p (Wilcoxon)": r["p (Wilcoxon)"], "n": r["n"]})
        fam = pd.DataFrame(fam); fam["p (Holm)"] = holm(fam["p (Wilcoxon)"]).values
        rows.append(fam)
    FAT = pd.concat(rows).set_index(["dataset", "contraste"])
    FAT.to_csv(OUT / "parte2_fatorial_contrastes.csv")
    display(FAT.round(3), nome="P2 fatorial contrastes")
    print("Δ negativo = B errou menos que A.")

    rows = []
    for ds in DATASETS:
        d = F[F.dataset == ds]
        fit = smf.ols("abs_err ~ C(versao, Treatment('original')) * cartao + C(issue_id)", data=d).fit(
            cov_type="cluster", cov_kwds={"groups": pd.factorize(d.issue_id)[0]})
        termos = [t for t in fit.params.index if "versao" in t or t == "cartao"]
        ci = fit.conf_int().loc[termos]
        for t in termos:
            legivel = (t.replace("C(versao, Treatment('original'))[T.", "texto ").replace("]", "")
                        .replace(":cartao", " × cartão").replace("cartao", "cartão"))
            rows.append({"dataset": ds, "termo": legivel, "coef. (SP)": fit.params[t], "IC95% inf": ci.loc[t, 0],
                         "IC95% sup": ci.loc[t, 1], "p": fit.pvalues[t], "n issues": d.issue_id.nunique()})
    REG = pd.DataFrame(rows).set_index(["dataset", "termo"])
    REG.to_csv(OUT / "parte2_fatorial_regressao.csv")
    display(REG.round(3), nome="P2 fatorial regressão")

    fig, axes = plt.subplots(1, len(DATASETS), figsize=(5.8 * len(DATASETS), 3.6))
    for ax, ds in zip(np.atleast_1d(axes), DATASETS):
        if ds not in tab.index.get_level_values(0):
            ax.set_visible(False); continue
        sub = tab.loc[ds]; xs = np.arange(len(sub))
        ax.plot(xs, sub["sem cartão"], "o-", color=COR["s1"], label="estimador sem cartão")
        ax.plot(xs, sub["com cartão"], "o-", color=COR["s2"], label="estimador com cartão")
        if ds in pm:
            ax.axhline(pm[ds], color=COR["muted"], linestyle="--", linewidth=1, label="mediana do projeto")
        ax.set_xticks(xs); ax.set_xticklabels(sub.index, rotation=18, fontsize=8)
        ax.set_ylabel("MAE (story points)"); ax.set_title(ds)
    np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.suptitle("Versão do texto × contexto no estimador (LLM), com a reescrita estruturada", fontweight="bold", color=COR["ink"])
    fig.tight_layout(); salva_fig("parte2_fatorial"); plt.show()

### 2f. As lacunas explicam o erro?

Se faltar informação na issue for o que atrapalha a estimativa, issues com mais campos "Not stated" deveriam ter
erro maior **no texto original**. Cuidado: é uma associação, e as lacunas também se relacionam com o tamanho e o
tipo da issue (como na seção 7 do `analise_jev.ipynb`); por isso a tabela traz também a correlação com o próprio SP.

In [ ]:
if PARTE2:
    base_err = EE[EE.versao == "original"].pivot_table(index="issue_id", columns="modelo", values="abs_err")
    LL = L.set_index("issue_id").join(base_err, how="inner")
    rows = []
    for ds in DATASETS:
        d = LL[LL.dataset == ds]
        for alvo in ["sp", *base_err.columns]:
            if d.n_lacunas.nunique() > 1 and len(d) >= 10:
                rho, p = stats.spearmanr(d.n_lacunas, d[alvo])
                rows.append({"dataset": ds, "n_lacunas ×": "SP real" if alvo == "sp" else f"|erro| {alvo} (texto original)",
                             "rho (Spearman)": rho, "p": p, "n": len(d)})
    if rows:
        display(pd.DataFrame(rows).set_index(["dataset", "n_lacunas ×"]).round(3), nome="P2 lacunas x erro")
    rows = []
    for ds in DATASETS:
        d = LL[LL.dataset == ds]
        for c in CAMPOS_LACUNA:
            a, b = d[~d[c]][tag0], d[d[c]][tag0]
            if len(a) >= 5 and len(b) >= 5:
                rows.append({"dataset": ds, "campo": c, "MAE LLM com o campo": a.mean(), "MAE LLM sem o campo": b.mean(),
                             "n com": len(a), "n sem": len(b),
                             "p (Mann-Whitney)": stats.mannwhitneyu(a, b).pvalue})
    if rows:
        display(pd.DataFrame(rows).set_index(["dataset", "campo"]).round(3), nome="P2 erro por campo ausente")

## Salvar

Grava todas as tabelas num Excel (`results/analise_extensoes/relatorio_extensoes.xlsx`); as figuras já ficaram em
`results/analise_extensoes/figuras/`.

In [ ]:
xlsx = OUT / "relatorio_extensoes.xlsx"
with pd.ExcelWriter(xlsx) as w:
    usados = set()
    for nome, t in TABELAS.items():
        aba = re.sub(r"[\[\]\*\?/\\:]", "", nome)[:31]
        k = 2
        while aba in usados:
            aba = f"{aba[:28]}_{k}"; k += 1
        usados.add(aba)
        t.to_excel(w, sheet_name=aba)
print(f"{len(TABELAS)} tabelas em {xlsx.relative_to(ROOT)}")